# Pikachu Monthly Price Mover Tracker

Finds the 10 Pikachu-named Pokémon cards whose Cardmarket price has moved the most (up or down) over roughly the past month.

**Ranking**: `(avg7 - avg30) / avg30` (Cardmarket, EUR) — a moving-average crossover, similar to comparing short vs. long moving averages for a stock. Selected into the top 10 by `abs(ln(avg7/avg30))` so gainers and droppers are compared fairly (raw % change is bounded at -100% but unbounded upward, which would otherwise skew the list toward gainers). Cards under €1.00 (30-day avg) are excluded as noise. This is an approximation, not the price exactly 30 days ago — see `README.md` for the full rationale and caveats.

Cards whose set released in the last 30 days get a blue **NEW!** badge.

In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd

from pikachu_core import get_top_movers

logging.basicConfig(level=logging.INFO, format="%(message)s")

In [ ]:
movers = get_top_movers(n=10)

df = pd.DataFrame(movers)
df.insert(0, "rank", range(1, len(df) + 1))
df["card"] = df["name"] + df["is_new"].map({True: "  🆕 NEW!", False: ""})

display_df = df[["rank", "card", "set", "pct_change_month", "pct_change_24h", "usd_display_price"]].rename(
    columns={
        "pct_change_month": "Monthly move %",
        "pct_change_24h": "Last 24h %",
        "usd_display_price": "USD ref. price",
        "card": "Card",
        "set": "Set",
        "rank": "#",
    }
)
display_df

In [ ]:
def _color_by_sign(val):
    if not isinstance(val, (int, float)):
        return ""
    color = "#1a7f37" if val >= 0 else "#cf222e"
    return f"color: {color}; font-weight: 600"


styled = (
    display_df.style
    .map(_color_by_sign, subset=["Monthly move %", "Last 24h %"])
    .format({
        "Monthly move %": "{:+.1f}%",
        "Last 24h %": lambda v: f"{v:+.1f}%" if pd.notna(v) else "n/a",
        "USD ref. price": lambda v: f"${v:,.2f}" if pd.notna(v) else "n/a",
    })
    .hide(axis="index")
)
styled

In [ ]:
plot_df = df.sort_values("pct_change_month")
colors = ["#1a7f37" if v >= 0 else "#cf222e" for v in plot_df["pct_change_month"]]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(plot_df["card"], plot_df["pct_change_month"], color=colors)
ax.axvline(0, color="#999", linewidth=0.8)
ax.set_xlabel("Monthly price move (%)")
ax.set_title("Top 10 Pikachu Price Movers — Monthly Change")
ax.bar_label(bars, fmt="%+.1f%%", padding=3, fontsize=9)
fig.tight_layout()
plt.show()